<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/Deep_Hedging_Model_Risk_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Validation & Benchmark Testing of Reinforcement Learning (RL) Deep Hedging Policies under Jump Diffusion & Proportional Frictions**

## **1. Executive Summary**

This document provides the formal validation framework for machine learning/reinforcement learning (ML/RL) automated option hedging agents. Per SR 11-7 / OCC 2011-12 Model Risk Management guidelines, complex non-linear hedging models must be benchmarked against robust, analytic baselines under non-Gaussian state dynamics and market frictions.

**Validation Objectives**
1. **Benchmark Baseline:** Benchmark the policy against a discrete-time continuous-rebalancing Black-Scholes Delta hedge.
2. **Stress Dynamic Injection:** Evaluate policy resilience under **Merton Jump Diffusion (MJD)** to account for heavy tail risk and jump-induced market gaps.
3. **Friction Drag:** Incorporate proportional transaction costs to quantify market impact and over-trading penalities.
4. **Tail Risk Quantification:** Decisioning based on Expected Shortfall ($\text{Expected Shortfall}_{97.5\%}$) to penalize severe left-tail PnL deficits over pure variance metrics.

## **2. Mathematical & Stochastic Foundations**

#### **2.1 Underlying Asset Dynamics: Merton Jump Diffusion (MJD)**

To test policy robustness against market regime shifts, the underlying asset process $S_{t}$ follows the Stochastic Differential Equation (SDE):

$$dS_{t} = (r-\lambda k)S_{t}-d_{t} + \sigma S_{t} - dW_{t} + (J-1)S_{t}-dN_{t}$$

Where:
* $W_{t}$ is a standard Brownian motion.
* $N_{t}$ is a Poisson process with jump intensity parameter $\lambda \geq 0$.
* $J$ is the random jump size, where $ln(J) \sim N(\mu_{J}, \sigma^{2}_{J})$.
* $k = E[J-1] = exp(\mu_{J} + \frac{1}{2}\sigma^{2}_{J})-1$ represents the expected relative jump size, compensating the risk-neutral drift.

---

#### **2.2 Discrete Portfolio PnL Accounting with Friction**

Let $\phi_{k} \in [0, 1]$ be the delta allocation held over the interval $[t_{k}, t_{k+1}]$. The cumulative portfolio cash account $C_{T}$ and overall pathwise terminal profit and loss (PnL) accounting is defined as:


$$\Pi_{T}= - \Sigma^{N-1}_{k=0}[(\phi_{k+1} - \phi_{k})S_{t_{k}} + c \cdot |\phi_{k+1} - \phi_{k}|S_{t_{k}}] + \phi_{N}S_{T} - max(S_{T} - K, 0)$$

where $c$ is the proportional transaction cost rate (e.g., 10 bps = 0.001).

---

#### **2.3 Risk-Adjusted Decision Metric: Expected Shortfall ($ES_{\alpha}$)**

Standard variance-based metrics (e.g., Sharpe Ratio) fail to capture asymmetric tail losses inherent in jump environments. We utilize Expected Shortfall at confidence level $1-\alpha = 97.5\%$.

$$\text{ES}_{\alpha}(\Pi) = -E[\Pi | \Pi \leq q_{\alpha}(\Pi)] = -\frac{1}{\alpha}\int^{\alpha}_{0}VaR_{u}(\Pi)du$$

An RL policy is rejected if its tail deficit exceeds the Black-Scholes benchmark by more than the regulatory tolerance threshold ($\kappa = 1.25$).

$$\text{Decision} =
\begin{cases}
\text{REJECT_POLICY}, & \text{if } \text{ES}_{97.5\%}^{\text{RL}} > 1.25 \times \text{ES}_{97.5\%}^{\text{BS}} \\
\text{APPROVED}, & \text{otherwise}
\end{cases}$$


In [3]:
"""
======================================================================================================
MODEL RISK MANAGEMENT QUANTITATIVE VALIDATION ENGINE
MODEL CLASS: Deep Hedging vs. Analytic Benchmark Validator
======================================================================================================
"""

import numpy as np
import scipy.stats as si
from typing import Dict, Callable, Tuple

class DeepHedgingValidator:
  """
  Validation engine comparing an Machine Learning/Reinforcement Learning hedging strategies
  against analytical benchmark Black-Scholes Delta model under Merton Jump Diffusion and transaction friction.
  """
  def __init__(self, S0: float, K: float, T: float, r: float, sigma: float, cost_rate: float = 0.001):
    self.S0 = float(S0)
    self.K = float(K)
    self.T = float(T)
    self.r = float(r)
    self.sigma = float(sigma)
    self.cost = float(cost_rate)

  def black_scholes_delta_vectorized(self, S: float, t: float) -> np.ndarray:
    """ Calculates vectorized European Call Delta across all path instances."""
    tau = np.maximum(self.T - t, 1e-6)
    d1 = (np.log(S / self.K) + (self.r + 0.5 * self.sigma ** 2) * tau) / (self.sigma * np.sqrt(tau))
    return si.norm.cdf(d1)

  def simulate_jump_diffusion_paths(
      self,
      n_paths: int = 5000,
      n_steps: int = 50,
      lambda_jump: float = 0.2,
      mu_jump: float = -0.05,
      jump_std: float = 0.1
      ) -> np.ndarray:
      """
      Simulates asset price trajectories following Merton Jump Diffusion (MJD)
      incorporating martingale drift compensation for the jump term.
      """
      dt = self.T / n_steps
      paths = np.zeros((n_paths, n_steps + 1))
      paths[:, 0] = self.S0

      # Compensated drift for jump process to maintain risk-neutral measure
      k = np.exp(mu_jump + 0.5 * jump_std ** 2) - 1.0
      compensated_drift = (self.r - lambda_jump * k - 0.5 * self.sigma ** 2) * dt
      diffusion_scale = self.sigma * np.sqrt(dt)

      for t in range(1, n_steps + 1):
        z = np.random.normal(0.0, 1.0, n_paths)

        # Compound Poisson process realization
        n_jumps = np.random.poisson(lambda_jump * dt, n_paths)
        jump_magnitudes = np.zeros(n_paths)

        # Vectorized sampling for path jump events
        mask = n_jumps > 0
        if np.any(mask):
          for i in np.where(mask)[0]:
            jump_magnitudes[i] = np.sum(np.random.normal(mu_jump, jump_std, n_jumps[i]))

      return paths

  def validate_policy_pnl(
      self,
      paths: np.ndarray,
      rl_policy_func: Callable[[np.ndarray, float], np.ndarray],
      es_alpha: float = 0.025
      ) -> Dict[str, float]:
      """
      Executes discrete pathwise PnL attribution, friction cost deduction,
      terminal payoff settlement, and Exected Shortfall evaluation.
      """
      n_paths, n_steps = paths.shape[0], paths.shape[1] - 1
      dt = self.T / n_steps

      bs_pnl = np.zeros(n_paths)
      rl_pnl = np.zeros(n_paths)

      bs_pos = np.zeros(n_paths)
      rl_pos = np.zeros(n_paths)

      for i in range(n_steps):
        t = i * dt
        S_t = paths[:, i]

        # 1. Evaluate Benchmark (Black-Scholes Delta)
        new_bs_pos = np.array([self.black_scholes_delta_vectorized(s, t) for s in S_t])
        bs_trade = new_bs_pos - bs_pos
        bs_pnl -= bs_trade * S_t + np.abs(bs_trade) * S_t * self.cost
        bs_pos = new_bs_pos

        # 2. Evaluate Candidate Policy (RL Agent)
        new_rl_pos = rl_policy_func(S_t, t)
        rl_trade = new_rl_pos - rl_pos
        rl_pnl -= rl_trade * S_t + np.abs(rl_trade) * S_t * self.cost
        rl_pos = new_rl_pos

      # Terminal Liquidation & Option Settlement
      terminal_S = paths[:, -1]
      payoff = np.maximum(terminal_S - self.K, 0.0)

      bs_total_pnl =  bs_pnl + bs_pos * terminal_S - payoff
      rl_total_pnl = rl_pnl + rl_pos * terminal_S - payoff

      # Risk Attribution: Tail Risk Metrics (97.5% ES)
      cutoff_idx = int(es_alpha * n_paths)

      bs_sorted_pnl = np.sort(bs_total_pnl)
      rl_sorted_pnl = np.sort(rl_total_pnl)

      bs_es = -np.mean(np.sort(bs_total_pnl)[:cutoff_idx])
      rl_es = -np.mean(np.sort(rl_total_pnl)[:cutoff_idx])

      # Model Governance Decision Matrix
      tail_deficit = rl_es - bs_es
      decision = "REJECT_POLICY" if rl_es > (bs_es * 1.25) else "APPROVED"

      return {
          "BS_Expected_Shortfall_97.5": float(bs_es),
          "RL_Expected_Shortfall_97.5": float(rl_es),
          "Tail_Risk_Deficit": float(tail_deficit),
          "ES_Ratio_RL_vs_BS": float(rl_es / bs_es) if bs_es != 0 else np.nan,
          "Validation_Decision": decision
      }

# ====================================================================================
# EXECUTABLE VERIFICATION SUITE
# ====================================================================================
if __name__ == "__main__":
  # Initialize Validator with standard parameter set
  validator = DeepHedgingValidator(
      S0=100,
      K=100,
      T=0.25,
      r=0.02,
      sigma=0.2,
      cost_rate=0.001   # 10 bps transaction drag
      )

  # Generate MJD Stress Scenario Data Set
  np.random.seed(42)
  stress_paths = validator.simulate_jump_diffusion_paths(
      n_paths=5000,
      n_steps=50,
      lambda_jump=0.3,
      mu_jump=-0.08,
      jump_std=0.12
  )

  # Candidate Sub-optimal / Un-tuned RL Policy Simulation
  def candidate_rl_policy(S: np.ndarray, t: float) -> np.ndarray:
    """
    Simulates an RL policy with imperfect learning convergence and execution noise.
    """
    bs_delta = validator.black_scholes_delta_vectorized(S, t)
    execution_noise = np.random.normal(0.0, 0.04, size=len(S))
    return np.clip(bs_delta + execution_noise, 0.0, 1.0)

  # Run Model RIsk Assessment
  metrics = validator.validate_policy_pnl(stress_paths, candidate_rl_policy)

  print("\n ---------------------------------------------------------------")
  print(" QUANTITATIVE MODEL RISK MANAGEMENT VALIDATION REPORT")
  print("\n ---------------------------------------------------------------")
  for key, val in metrics.items():
    if isinstance(val, float):
      print(f"  {key:<30}: {val:12.5f}")
    else:
      print(f"  {key:<30}: {val}")
  print("\n ---------------------------------------------------------------")


/tmp/ipykernel_832/1667028386.py:28: RuntimeWarning: divide by zero encountered in log
  d1 = (np.log(S / self.K) + (self.r + 0.5 * self.sigma ** 2) * tau) / (self.sigma * np.sqrt(tau))



 ---------------------------------------------------------------
 QUANTITATIVE MODEL RISK MANAGEMENT VALIDATION REPORT

 ---------------------------------------------------------------
  BS_Expected_Shortfall_97.5    :     54.03677
  RL_Expected_Shortfall_97.5    :     63.63103
  Tail_Risk_Deficit             :      9.59426
  ES_Ratio_RL_vs_BS             :      1.17755
  Validation_Decision           : APPROVED

 ---------------------------------------------------------------


## **3. Model Limitations**

**Structural Vulnerabilities in Baseline Implementation**
1. **Interest Rate Accumulation:** The validation engine treats cash balances as uninvested static balances. In high interest-rate environment, the cash position needs explicit time-value discounting to avoid underestimating hedging carrying costs.
2. **Missing Gamma Drag:** Under Poisson jump conditions, Delta-hedging alone cannot mitigate discrete price gaps ($\Delta S_{t} > 0$). To pass Model Risk Management checks for desk deployment, candidate deep hedging models must be granted access to additional instruments (e.g., short-dated variance swaps or proxy options) to internalize Gamma/Vega risk.
3. **Execution Lag and Microstructure Frictions:** The current engine assumes instantaneous execution at $S_{t}$. Real-world deployment requires modeling bid-ask spread asymmetry and price impact functions.